In [ ]:
%load_ext autoreload
%autoreload 2  

In [ ]:
# Standard imports
import pandas as pd
import numpy as np
import sys
import os
import matplotlib.pyplot as plt
import zarr
import dask.array as da
from tqdm import tqdm  # For progress bars

# Add path to your analysis code
pythonPackagePath = os.path.abspath(r'D:\Akamatsu_Lab\LLSM-CME-ANALYSIS\Final\src')
sys.path.append(pythonPackagePath)

# Import your existing track functions
from filters import Track, create_tracks_from_dataframe, drop_short_tracks
from filters import drop_early_peak_tracks, drop_last_frame_peak_tracks, drop_tracks_below_intensity

# Import the buffer analysis library we just created
from llsm_buffer_analysis import (
    classify_tracks, 
    process_track_buffers,
    determine_track_validity,
    cleanup_matlab_engine,
    fit_all_track_frames,
    extract_fit_parameters_to_dataframe,
    get_track_intensity_profiles,
    BUFFER_FRAMES, 
    SIGMA_VALUES
)

# Set up MATLAB paths if needed
import matlab.engine
print("Initializing MATLAB engine...")
eng = matlab.engine.start_matlab()

# IMPORTANT: Update this path to your llsmtools location
LLSMTOOLS_PATH = r'D:\Akamatsu_Lab\llsmtools'  # <-- UPDATE THIS!
eng.addpath(os.path.join(LLSMTOOLS_PATH, 'psdetect3d'), nargout=0)
eng.addpath(os.path.join(LLSMTOOLS_PATH, 'iofunc'), nargout=0)
print("MATLAB engine ready")

# Set plotting defaults
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 600
plt.rcParams['font.size'] = 12

In [ ]:
# ============== USER CONFIGURATION ==============

# Data paths
BASE_DIR = r'Z:\Abhi\LLSM_Analysis'
INPUT_FILE_DIR = '41_OS_analysis/'
ZARR_FILE_DIR = '41_OS_analysis/zarr_file/all_channels_data'

# Analysis parameters
THRESHOLD_LENGTH = 3  # Minimum track length in frames
PEAK_CUTOFF = 3  # Drop tracks with peak at or before this frame
CHANNEL_DETECTED = 3  # Channel used for detection (AP2)

# Channel configuration
CHANNEL_NAMES = {1: 'ARPC3', 2: 'DNM2', 3: 'AP2'}
CHANNEL_COLORS = {'AP2': 'red', 'DNM2': 'green', 'ARPC3': 'blue'}

# Buffer analysis parameters (can override defaults from library)
BUFFER_FRAMES = 5
CONFIDENCE_LEVEL = 0.95

# Output options
SAVE_PLOTS = False
SAVE_INTERMEDIATE = True

# ================================================

# Construct full paths
zarr_path = os.path.join(BASE_DIR, ZARR_FILE_DIR)
input_pkl_path = os.path.join(BASE_DIR, INPUT_FILE_DIR, 'archive/track_df_cleaned_final_full.pkl')
output_dir = os.path.join(BASE_DIR, INPUT_FILE_DIR, 'datasets/track_classification_results')

# Create output directory if needed
os.makedirs(output_dir, exist_ok=True)

# Define output files
output_files = {
    'complete': os.path.join(output_dir, 'complete_tracks_with_buffers.pkl'),
    'partial': os.path.join(output_dir, 'partial_tracks.pkl'),
    'persistent': os.path.join(output_dir, 'persistent_tracks.pkl'),
    'invalid': os.path.join(output_dir, 'invalid_tracks.pkl'),
    'summary': os.path.join(output_dir, 'classification_summary.csv'),
    'prefilter': os.path.join(output_dir, 'prefilter_tracks.pkl')
}

print("Configuration loaded successfully")
print(f"Input data: {input_pkl_path}")
print(f"Movie data: {zarr_path}")
print(f"Output directory: {output_dir}")

In [ ]:
# Load track dataframe
track_df = pd.read_pickle(input_pkl_path)
z2 = zarr.open(zarr_path, mode='r')
movie_shape = z2.shape

In [ ]:
#Trial
track_df = track_df.sort_values(by=['track_id', 'frame'])
# Subset all rows with track_id between 0 and 5 (inclusive)
# track_df_trial = track_df[track_df['track_id'].between(20000,20019)].copy()
track_id = 20016
track_df_trial = track_df[track_df['track_id'].between(track_id, track_id)].copy()

In [ ]:
track_df_trial[track_df_trial['track_id'] == track_id]
# track_df[track_df['track_id'] == 27375]
# track_df[
#     (track_df['frame'] == 51) &
#     (track_df['mu_x'].between(26, 46)) &
#     (track_df['mu_y'].between(1262, 1282)) &
#     (track_df['mu_z'].between(0, 20))
# ]
# track_df[
#     (track_df['frame'].isin([78, 79, 80, 81, 82])) &
#     (track_df['mu_x'].between(25, 45)) &
#     (track_df['mu_y'].between(1263, 1283)) &
#     (track_df['mu_z'].between(2, 22))
# ]

In [ ]:
tracks = create_tracks_from_dataframe(df = track_df_trial, intensities_col_name = ['c3_peak_mean', 'c2_peak_mean', 'c1_peak_mean'], 
track_id_col_name = 'track_id', frame_col_name = 'frame', coords = ['mu_x', 'mu_y', 'mu_z'], trackability_col_name = 'segTrackability')

In [ ]:
# Classify tracks
classified_tracks = classify_tracks(
    tracks, 
    z2, 
    buffer_frames=BUFFER_FRAMES,
    ap2_channel_idx=2  # Channel 3 (AP2) is index 2
)

print("\n" + "="*60)
print("CLASSIFICATION RESULTS")
print("="*60)

# Create summary statistics
summary_data = []
total_tracks = len(tracks)

for category, track_list in classified_tracks.items():
    n_tracks = len(track_list)
    percentage = (n_tracks / total_tracks * 100) if total_tracks > 0 else 0
    
    summary_data.append({
        'Category': category.upper(),
        'Count': n_tracks,
        'Percentage': percentage
    })
    
    print(f"\n{category.upper()} TRACKS:")
    print(f"  Count: {n_tracks} ({percentage:.1f}%)")
    
    if n_tracks > 0 and track_list[0] is not None:
        # Calculate additional statistics
        lengths = [t.track_length for t in track_list if hasattr(t, 'track_length')]
        if lengths:
            print(f"  Mean length: {np.mean(lengths):.1f} ± {np.std(lengths):.1f} frames")
            print(f"  Length range: {np.min(lengths)}-{np.max(lengths)} frames")

# Create and save summary DataFrame
summary_df = pd.DataFrame(summary_data)
print("\n" + "="*60)
print("SUMMARY TABLE")
print("="*60)
print(summary_df.to_string(index=False))

if SAVE_INTERMEDIATE:
    summary_df.to_csv(output_files['summary'], index=False)
    print(f"\nSaved summary to {output_files['summary']}")

#### WHAT SHOULD THE VALUE OF SIGMA BE?? CURRENTLY SET TO 2.25! ####
#### Why does the value of the estimate depend on the number of buffer frames?? ####
#### What should the value of window_size be? ####

In [ ]:
# Save each category of tracks
print("\n" + "="*60)
print("SAVING CLASSIFIED TRACKS")
print("="*60)

for category, track_list in classified_tracks.items():
    if len(track_list) == 0:
        print(f"No {category} tracks to save")
        continue
    
    # Create DataFrame for this category
    category_data = []
    
    for track in tqdm(track_list, desc=f"Processing {category} tracks"):
        track_dict = {
            'track_id': track.track_id.values[0] if hasattr(track.track_id, 'values') else track.track_id,
            'track_length': track.track_length if hasattr(track, 'track_length') else len(track.frame),
            'track_start': track.track_start if hasattr(track, 'track_start') else min(track.frame),
            'track_end': track.track_end if hasattr(track, 'track_end') else max(track.frame),
            'classification': category
        }
        
        # Add coordinates
        if hasattr(track, 'x'):
            track_dict['x_coords'] = track.x
            track_dict['y_coords'] = track.y
            track_dict['z_coords'] = track.z
            track_dict['mean_x'] = np.mean(track.x) if hasattr(track.x, '__iter__') else track.x
            track_dict['mean_y'] = np.mean(track.y) if hasattr(track.y, '__iter__') else track.y
            track_dict['mean_z'] = np.mean(track.z) if hasattr(track.z, '__iter__') else track.z
        
        # Add intensity data
        if hasattr(track, 'peak_intensities'):
            for i, ch_name in CHANNEL_NAMES.items():
                track_dict[f'{ch_name}_peak'] = track.peak_intensities[i-1]
                track_dict[f'{ch_name}_peak_frame'] = track.peak_intensity_frames[i-1]
        
        # Add buffer analysis results if available
        if hasattr(track, 'buffer_results'):
            # Store buffer validity
            track_dict['passed_buffer_test'] = determine_track_validity(track, ap2_channel_idx=2)
            
            # Store buffer intensities for AP2 channel
            ap2_idx = 2
            start_buffer = track.buffer_results['start_buffer']
            end_buffer = track.buffer_results['end_buffer']
            
            # Extract intensities
            start_intensities = [b['A'] if b else np.nan for b in start_buffer]
            end_intensities = [b['A'] if b else np.nan for b in end_buffer]
            
            track_dict['start_buffer_intensities'] = start_intensities
            track_dict['end_buffer_intensities'] = end_intensities
            track_dict['start_buffer_mean'] = np.nanmean(start_intensities) if start_intensities else np.nan
            track_dict['end_buffer_mean'] = np.nanmean(end_intensities) if end_intensities else np.nan
        
        category_data.append(track_dict)
    
    # Create and save DataFrame
    category_df = pd.DataFrame(category_data)
    
    # Save to pickle
    output_path = output_files.get(category, os.path.join(output_dir, f'{category}_tracks.pkl'))
    category_df.to_pickle(output_path)
    print(f"Saved {len(category_df)} {category} tracks to {output_path}")

In [ ]:
# Example: Examine specific tracks from each category
print("\n" + "="*60)
print("EXAMPLE TRACK ANALYSIS")
print("="*60)

def display_track_info(track, category):
    """
    Display track information with corrected buffer validation.
    """
    track_id = track.track_id.values[0] if hasattr(track.track_id, 'values') else track.track_id
    print(f"\n{category.upper()} Track ID: {track_id}")
    print(f"  Length: {track.track_length if hasattr(track, 'track_length') else 'N/A'}")
    print(f"  Frames: {track.track_start if hasattr(track, 'track_start') else 'N/A'} - "
          f"{track.track_end if hasattr(track, 'track_end') else 'N/A'}")
    
    if hasattr(track, 'peak_intensities'):
        CHANNEL_NAMES = {1: 'ARPC3', 2: 'DNM2', 3: 'AP2'}  # Define if not already
        for i, ch_name in CHANNEL_NAMES.items():
            print(f"  {ch_name} peak: {track.peak_intensities[i-1]:.1f} "
                  f"(frame {track.peak_intensity_frames[i-1]})")
    
    if hasattr(track, 'buffer_results'):
        # Use corrected validation function with verbose output
        passed = determine_track_validity(track, ap2_channel_idx=2, verbose=True)
        print(f"  Buffer test: {'✅ PASSED' if passed else '❌ FAILED'}")
        
        # Show buffer statistics
        start_buffer = track.buffer_results['start_buffer']
        end_buffer = track.buffer_results['end_buffer']
        
        # Calculate mean and max for both A and A+c
        start_A = [b['A'] for b in start_buffer if b is not None and 'A' in b]
        start_Ac = [b['A'] + b['c'] for b in start_buffer if b is not None and 'A' in b and 'c' in b]
        
        end_A = [b['A'] for b in end_buffer if b is not None and 'A' in b]
        end_Ac = [b['A'] + b['c'] for b in end_buffer if b is not None and 'A' in b and 'c' in b]
        
        if start_A:
            print(f"  Start buffer: A mean={np.mean(start_A):.1f}, max={np.max(start_A):.1f}, "
                  f"A+c mean={np.mean(start_Ac):.1f}, max={np.max(start_Ac):.1f}")
        if end_A:
            print(f"  End buffer:   A mean={np.mean(end_A):.1f}, max={np.max(end_A):.1f}, "
                  f"A+c mean={np.mean(end_Ac):.1f}, max={np.max(end_Ac):.1f}")


# Show examples from each category
for category, track_list in classified_tracks.items():
    if len(track_list) > 0:
        # Show first track as example
        display_track_info(track_list[0], category)

#### What are the start and end buffer means? Why do they always seem to be the same, and = 3.0?

In [ ]:
#### Running Gaussian Spot fitting on an entire track ####
# Option 2: Fit only AP2 channel (channel 3 = index 2)
track_fits = fit_all_track_frames(
    tracks,  # Just fit first 10 tracks for testing
    z2,
    buffer_frames=BUFFER_FRAMES,
    channels_to_fit=[2]  # Only AP2
)

In [ ]:
# Convert to DataFrame for analysis
#### Is the code defaulting to the Python fallback because MATLAB is not working?? #### Why are some of the standard errors NaN??
df = extract_fit_parameters_to_dataframe(track_fits, channel_idx=2)
df[df['track_id'] == track_id]

In [ ]:
def plot_track_aguet_style(df, track_id, k_level=1.96, figsize=(10, 6)):
    """
    Plot in the Aguet paper style with error bands.
    
    Parameters:
    -----------
    df : pandas DataFrame
        DataFrame with columns: track_id, frame_idx, frame_type, A, A_pstd, sigma_r
    track_id : int or str
        Track ID to plot
    k_level : float
        Confidence level multiplier (default: 1.96 for 95% confidence)
    figsize : tuple
        Figure size (width, height)
    
    Returns:
    --------
    fig, ax : matplotlib figure and axis objects
    """
    # Filter data for this track
    track_data = df[df['track_id'] == track_id].copy()
    
    if len(track_data) == 0:
        print(f"No data found for track {track_id}")
        return None, None
    
    # Sort by frame index
    track_data = track_data.sort_values('frame_idx').reset_index(drop=True)
    
    # Extract data
    frames = track_data['frame_idx'].values
    A = track_data['A'].values
    A_pstd = track_data['A_pstd'].values
    sigma_r = track_data['sigma_r'].values
    frame_types = track_data['frame_type'].values
    
    # Create figure
    fig, ax = plt.subplots(figsize=figsize)
    
    # Significance threshold (k_level * sigma_r)
    significance_threshold = sigma_r * k_level
    
    # Calculate error bounds for intensity
    upper_bound = A + A_pstd
    lower_bound = A - A_pstd
    
    # Find track boundaries
    track_mask = frame_types == 'track'
    if np.any(track_mask):
        track_start_idx = np.where(track_mask)[0][0]
        track_end_idx = np.where(track_mask)[0][-1]
        track_start = frames[track_start_idx]
        track_end = frames[track_end_idx]
    else:
        track_start_idx = 0
        track_end_idx = len(frames) - 1
        track_start = frames[0]
        track_end = frames[-1]
    
    # Shade significance threshold regions
    # Left buffer (RED)
    if track_start_idx > 0:
        left_frames = frames[:track_start_idx+1]
        left_threshold = significance_threshold[:track_start_idx+1]
        ax.fill_between(left_frames, 0, left_threshold,
                         color='#e74c3c', alpha=0.3, zorder=1)
    
    # Right buffer (RED)
    if track_end_idx < len(frames) - 1:
        right_frames = frames[track_end_idx:]
        right_threshold = significance_threshold[track_end_idx:]
        ax.fill_between(right_frames, 0, right_threshold,
                         color='#e74c3c', alpha=0.3, zorder=1)
    
    # Track region (GREEN)
    track_frames = frames[track_start_idx:track_end_idx+1]
    track_threshold = significance_threshold[track_start_idx:track_end_idx+1]
    ax.fill_between(track_frames, 0, track_threshold,
                     color='#2ecc71', alpha=0.2,
                     label=f'Significance threshold ({k_level}×σ_r)', zorder=1)
    
    # Mark track boundaries with vertical dashed lines
    ax.axvline(track_start - 0.5, color='black', linestyle='--', 
               linewidth=1.5, alpha=0.7, zorder=2)
    ax.axvline(track_end + 0.5, color='black', linestyle='--', 
               linewidth=1.5, alpha=0.7, zorder=2)
    
    # Plot error band for intensity (continuous green band)
    ax.fill_between(frames, lower_bound, upper_bound,
                     color='#27ae60', alpha=0.25,
                     label='±1 s.d. (intensity)', zorder=2)
    
    # Plot intensity above background (A) line
    ax.plot(frames, A, 'o-', color='#27ae60', 
            linewidth=2.5, markersize=7, alpha=0.9,
            label='Intensity above background', zorder=3)
    
    # Labels
    ax.set_xlabel('Frame', fontsize=12)
    ax.set_ylabel('Fluorescence Intensity (A.U.)', fontsize=12)
    ax.set_title(f'Track {track_id}', fontsize=14, fontweight='bold')
    
    # Legend in upper left
    ax.legend(loc='upper left', fontsize=10)
    
    # Set y-axis to start at 0
    ax.set_ylim(bottom=0)
    
    plt.tight_layout()
    
    return fig, ax

In [ ]:
# Or simpler Aguet style
fig, ax = plot_track_aguet_style(df, track_id=track_id)
# plt.savefig(f'D:/Akamatsu_Lab/LLSM-CME-ANALYSIS/Final/Jupyter_Notebooks/matlab_interface/Abhi_figures/track_fragments/track_{track_id}_aguet_style.png', dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
# Get intensity profiles
profiles = get_track_intensity_profiles(track_fits, channel_idx=2)
for track_id, profile in list(profiles.items())[:3]:
    print(f"\nTrack {track_id}:")
    print(f"  Frames: {profile['frame_indices']}")
    print(f"  Types: {profile['frame_types']}")
    print(f"  Intensities: {profile['intensities']}") #### This seems to be A + c, not just A? ####

In [ ]:
# # Import the debug functions
# from debug_matlab_structure import quick_test_with_real_data, print_suggested_fix

# # Test with one of your classified tracks
# result = quick_test_with_real_data(
#     track=tracks[0],  # or any track
#     movie_data=z2,
#     channel_idx=2  # AP2 channel
# )

# # This will print detailed diagnostics showing:
# # 1. How MATLAB returns the 'res' structure
# # 2. Which access method works for 'std' and 'hAD'
# # 3. The exact values extracted

In [ ]:
# Close MATLAB engine when done
print("\nCleaning up...")
cleanup_matlab_engine()
print("Analysis complete!")

# Final summary
print("\n" + "="*60)
print("FINAL OUTPUT FILES")
print("="*60)
for name, path in output_files.items():
    if os.path.exists(path):
        size = os.path.getsize(path) / 1024  # Size in KB
        print(f"{name:15} {size:>10.1f} KB  {path}")